# **Customer Trends Analysis**

> The goal of this project is to understand customers' shopping behavior in order to improve sales, customer satisfaction and long-term loyalty. This Python part is to clean and transform the raw dataset for analysis.




## **1. Data Ingestion**

In [1]:
# Authenticate user and connect to Bigquery
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

project_id = 'customer-trends-analysis'
client = bigquery.Client(project=project_id)

In [2]:
# Get the customer dataset and table
import pandas as pd

dataset_ref = client.dataset('customer', project=project_id)

customer_table = client.get_table(dataset_ref.table('customer_behavior'))
df = client.list_rows(table=customer_table).to_dataframe()

df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount USD,Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,106,69,Male,Backpack,Accessories,96,New York,M,Charcoal,Fall,3.6,True,Next Day Air,True,True,4,Bank Transfer,Annually
1,133,30,Male,Backpack,Accessories,58,New Mexico,M,Orange,Fall,4.7,True,2-Day Shipping,True,True,42,PayPal,Weekly
2,162,65,Male,Backpack,Accessories,49,North Dakota,M,Olive,Fall,3.0,True,Express,True,True,29,Venmo,Fortnightly
3,381,39,Male,Backpack,Accessories,69,New Hampshire,XL,Beige,Fall,3.0,True,Store Pickup,True,True,19,Bank Transfer,Weekly
4,403,31,Male,Backpack,Accessories,78,California,L,Cyan,Fall,3.0,True,Express,True,True,26,PayPal,Monthly


## **2. Data cleaning**

In [3]:
# Data types checking
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   Int64  
 1   Age                     3900 non-null   Int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount USD     3900 non-null   Int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   boolean
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   boolean
 14  Promo Code Used         3900 non-null   

In [4]:
df.describe()

,Customer ID,Age,Purchase Amount USD,Review Rating,Previous Purchases
count,3900.0,3900.0,3900.0,3863.000000,3900.0
mean,1950.5,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.0,18.0,20.0,2.500000,1.0
25%,975.75,31.0,39.0,3.100000,13.0
50%,1950.5,44.0,60.0,3.800000,25.0
75%,2925.25,57.0,81.0,4.400000,38.0
max,3900.0,70.0,100.0,5.000000,50.0


In [5]:
# Checking for null values
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount USD,0
Location,0
Size,0
Color,0
Season,0


=> Imput missing values in Review Rating column with the median review rating.

In [6]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [7]:
# Renaming columns
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ','_')
df = df.rename(columns={'purchase_amount_usd':'purchase_amount'})

In [8]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [9]:
# Create a new categorical column for age
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [10]:
# Create a new categorical column for frequency
frequency = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency)

In [11]:
# Checking for duplicated columns
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,True,True
1,True,True
2,True,True
3,True,True
4,True,True
5,True,True
6,True,True
7,True,True
8,True,True
9,True,True


In [12]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [13]:
# Dropping promo code used column
df = df.drop('promo_code_used', axis=1)

## **3. Load cleaned data into Bigquery**

In [14]:
df.to_gbq('customer.customer_trends', project_id, if_exists='replace')

/tmp/ipykernel_508/2482169061.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_gbq except for the argument 'destination_table' will be keyword-only.
  df.to_gbq('customer.customer_trends', project_id, if_exists='replace')
/tmp/ipykernel_508/2482169061.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df.to_gbq('customer.customer_trends', project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 6223.00it/s]
